# BR-CP: recover original benchmarks and export event-local evidence

This notebook scans **your authorized Google Drive** for original THUML CSVs and ZIPs; if missing, it attempts to download public originals from THUML/Hugging Face. It records SHA-256 and runs *only* descriptive event-local diagnostics using the model cached in the archived replication package. It **does not** prove conditional coverage or claim BR-CP detects either external event. Run all cells in Google Colab, granting Drive access in the prompt. Results are written to `MyDrive/BRCP_Recovery_2026-11-10` without modifying any existing files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, zipfile, json, hashlib, requests, shutil, subprocess, base64, io
ROOT=Path('/content/brcp_recovery');ROOT.mkdir(exist_ok=True)
OUTPUT=Path('/content/drive/MyDrive/BRCP_Recovery_2026-11-10');OUTPUT.mkdir(parents=True,exist_ok=True)
DATA=ROOT/'Class-1';DATA.mkdir(exist_ok=True)
RELS=['ETT-small/ETTh1.csv','ETT-small/ETTh2.csv','ETT-small/ETTm1.csv','ETT-small/ETTm2.csv','electricity/electricity.csv','traffic/traffic.csv','weather/weather.csv','exchange_rate/exchange_rate.csv','illness/national_illness.csv']
print('Output folder:',OUTPUT)

## 1 — Find files in Drive, including inside ZIP archives (read-only discovery)

In [ ]:
# Prefer exact matching basenames. Never treat a filename as proof of provenance.
from collections import defaultdict
wanted={Path(x).name:x for x in RELS}
found=defaultdict(list);archives=[];scanned=0
for walkroot in [Path('/content/drive/MyDrive'),Path('/content/drive/Shareddrives')]:
 if not walkroot.exists():continue
 for base,dirs,files in os.walk(walkroot):
  dirs[:]=[d for d in dirs if d not in {'node_modules','.git','.Trash','__pycache__'}]
  for f in files:
   scanned+=1; p=Path(base)/f
   if f in wanted:found[f].append(str(p))
   elif f.lower().endswith('.zip') and p.stat().st_size<4_000_000_000:archives.append(p)
print('Scanned',scanned,'files; raw matches',dict(found),'ZIPs to inspect',len(archives))
for zp in archives:
 try:
  with zipfile.ZipFile(zp) as z:
   for member in z.namelist():
    name=Path(member).name
    if name in wanted:found[name].append(f'ZIP::{zp}::{member}')
 except (OSError,zipfile.BadZipFile):pass
print('Including ZIPs:',{k:len(v) for k,v in found.items()})

## 2 — Recover each CSV from Drive first, otherwise obtain official THUML originals

In [ ]:
status=[];base='https://huggingface.co/datasets/thuml/Time-Series-Library'
for rel in RELS:
 name=Path(rel).name;dst=DATA/rel;dst.parent.mkdir(parents=True,exist_ok=True)
 provenance=None;error=''
 for src in found.get(name,[]):
  try:
   if src.startswith('ZIP::'):
    _,zip_path,member=src.split('::',2)
    with zipfile.ZipFile(zip_path) as z, z.open(member) as fin, dst.open('wb') as fout:shutil.copyfileobj(fin,fout)
   else:shutil.copyfile(src,dst)
   if dst.stat().st_size<100:raise ValueError('Too small')
   provenance='drive:'+src;break
  except Exception as e:error=str(e)
 if provenance is None:
  url=f'{base}/resolve/main/{rel}?download=true'
  for attempt in range(3):
   try:
    r=requests.get(url,timeout=(20,300),stream=True);r.raise_for_status()
    with dst.open('wb') as fout:
     for chunk in r.iter_content(1024*1024):
      if chunk:fout.write(chunk)
    if dst.stat().st_size<100:raise ValueError('Too small')
    provenance='official_THUML:'+url;break
   except Exception as e:error=repr(e)
 if not provenance:print('UNAVAILABLE:',rel,error);status.append({'relative_path':rel,'status':'missing','error':error});continue
 digest=hashlib.sha256(dst.read_bytes()).hexdigest()
 status.append({'relative_path':rel,'status':'found','provenance':provenance,'sha256':digest,'bytes':dst.stat().st_size})
 print('OK:',rel,dst.stat().st_size,'bytes',digest[:16])
(OUTPUT/'recovery_manifest.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
print('Complete',sum(x['status']=='found' for x in status),'of',len(status))

## 3 — Verify columns, record dates, and export a reproducible baseline

In [ ]:
import pandas as pd
checks=[]
for item in status:
 if item['status']!='found':continue
 df=pd.read_csv(DATA/item['relative_path']);row={'file':item['relative_path'],'nrows':len(df),'ncols':len(df.columns),'columns':list(df.columns)}
 if 'date' in df:
  dates=pd.to_datetime(df['date'],errors='coerce',format='mixed');row.update({'first_date':str(dates.min()),'last_date':str(dates.max()),'invalid_dates':int(dates.isna().sum())})
 checks.append(row)
(OUTPUT/'source_validation.json').write_text(json.dumps(checks,indent=2),encoding='utf-8')
print(pd.DataFrame(checks)[['file','nrows','ncols']].to_string(index=False))

## 4 — Restore the audited replication source and compute local Lehman/H1N1 traces

In [ ]:
PAYLOAD_B64 = '...'

In [ ]:
import base64,zipfile,os,subprocess,sys,shutil
# The preceding cell stores an embedded, compressed copy of the historic source
# plus the frozen regression forecaster. It has not been silently substituted.
PAYLOAD=base64.b64decode(PAYLOAD_B64)
with zipfile.ZipFile(io.BytesIO(PAYLOAD)) as z:z.extractall(ROOT)
work=ROOT/'BRCP_Replication';(work/'results').mkdir(parents=True,exist_ok=True);env=os.environ.copy();env['BRCP_DATA']=str(DATA)
if all((DATA/p).exists() for p in ['exchange_rate/exchange_rate.csv','illness/national_illness.csv']):
 for program in ['scripts/run_case_studies.py','event_local_diagnostics.py']:
  p=subprocess.run([sys.executable,program],cwd=str(work),env=env,capture_output=True,text=True,timeout=900)
  print(program,'exit code',p.returncode,'stdout:',p.stdout[-3000:],'stderr:',p.stderr[-1500:])
  if p.returncode:raise RuntimeError(program+' failed. Logs above.')
 shutil.copytree(work/'event_local_output',OUTPUT/'event_local_output',dirs_exist_ok=True)
 shutil.copy2(work/'results/real_case_studies.csv',OUTPUT/'real_case_studies_rerun.csv') if (work/'results/real_case_studies.csv').exists() else None
else:
 print('Raw event CSV missing. No event-local results have been created.')
# preserve original source archive and all downloaded data for complete replication
shutil.copytree(DATA, OUTPUT/'Class-1',dirs_exist_ok=True)
print('Results saved to',OUTPUT)

**Bring results back to ChatGPT:** Once the notebook finishes, share the Drive folder `BRCP_Recovery_2026-11-10` or upload the resulting `event_local_output` and `recovery_manifest.json`. The assistant must validate actual output before writing manuscript claims.